In [11]:
import os, logging, sys, cv2
import mediapipe as mp
from datetime import datetime

sys.path.append(os.path.abspath(".."))

from utils import save_landmarks_to_csv, get_landmarks_from_image, Landmark

In [5]:
def audit_without_modifying(output_root):
    subconjuntos = ['test', 'train_val']
    
    print("\n" + "="*60)
    print("AUDITORÍA DE SINCRONIZACIÓN (MODO LECTURA)")
    print("="*60)

    for sub in subconjuntos:
        sub_path = os.path.join(output_root, sub)
        if not os.path.exists(sub_path):
            print(f"\n[!] Subconjunto {sub}: No encontrado.")
            continue

        vistas = sorted([d for d in os.listdir(sub_path) if os.path.isdir(os.path.join(sub_path, d))])
        if not vistas:
            print(f"\n[!] Subconjunto {sub}: No contiene carpetas de vistas.")
            continue

        # Diccionario para guardar qué timestamps tiene cada vista
        # Estructura: { '00_15': {'16802_rgb', '16803_rgb'}, '00_16': {...} }
        vistas_data = {}

        print(f"\n Analizando {sub.upper()}...")
        
        for vista in vistas:
            vista_dir = os.path.join(sub_path, vista)
            timestamps_en_vista = set()
            
            for f in os.listdir(vista_dir):
                if f.endswith('.csv'):
                    # Extraemos el timestamp ignorando el prefijo de cámara
                    # '00_15_1680259460105_rgb.csv' -> '1680259460105_rgb.csv'
                    partes = f.split('_')
                    if len(partes) >= 3:
                        ts_id = "_".join(partes[2:])
                        timestamps_en_vista.add(ts_id)
            
            vistas_data[vista] = timestamps_en_vista
            print(f"   -> Vista {vista}: detectados {len(timestamps_en_vista)} timestamps únicos.")

        # 1. Calculamos la intersección (Timestamps que están en TODAS las vistas)
        common_timestamps = set.intersection(*vistas_data.values())
        
        # 2. Informar resultados
        print(f"\n RESULTADOS PARA {sub.upper()}:")
        print(f"   - Timestamps comunes (sincronizados): {len(common_timestamps)}")
        
        if len(common_timestamps) == 0:
            print("   [ALERTA] No hay coincidencia entre vistas. Los prefijos impiden la sincronización.")
        
        # 3. Detalle de huérfanos por vista
        for vista in vistas:
            total_vista = len(vistas_data[vista])
            huerfanos = total_vista - len(common_timestamps)
            if huerfanos > 0:
                print(f"   - Vista {vista}: tiene {huerfanos} archivos que sobran (no tienen pareja en otras vistas).")
                # Mostrar ejemplo de 3 nombres si hay discrepancia
                ejemplos = list(vistas_data[vista] - common_timestamps)[:3]
                print(f"     Ejemplo de timestamp huérfano: {ejemplos}")
            else:
                print(f"   - Vista {vista}: Está perfecta.")

    print("\n" + "="*60)
    print("FIN DE LA AUDITORÍA (No se han realizado cambios en los archivos)")
    print("="*60)

In [6]:
audit_without_modifying("Output_CSVs_backup/")


AUDITORÍA DE SINCRONIZACIÓN (MODO LECTURA)

 Analizando TEST...
   -> Vista 00_15: detectados 3153 timestamps únicos.
   -> Vista 00_16: detectados 3151 timestamps únicos.
   -> Vista 00_17: detectados 3155 timestamps únicos.

 RESULTADOS PARA TEST:
   - Timestamps comunes (sincronizados): 3149
   - Vista 00_15: tiene 4 archivos que sobran (no tienen pareja en otras vistas).
     Ejemplo de timestamp huérfano: ['1680259688934_rgb.csv', '1680259710309_rgb.csv', '1680259563995_rgb.csv']
   - Vista 00_16: tiene 2 archivos que sobran (no tienen pareja en otras vistas).
     Ejemplo de timestamp huérfano: ['1680262707155_rgb.csv', '1680262517197_rgb.csv']
   - Vista 00_17: tiene 6 archivos que sobran (no tienen pareja en otras vistas).
     Ejemplo de timestamp huérfano: ['1680262517197_rgb.csv', '1680262707155_rgb.csv', '1680259710309_rgb.csv']

 Analizando TRAIN_VAL...
   -> Vista 00_15: detectados 15162 timestamps únicos.
   -> Vista 00_16: detectados 15029 timestamps únicos.
   -> Vist

In [4]:
!tree Output_CSVs_backup > tree.txt

In [7]:
import os

def sync_and_clean_interactive(output_root):
    subconjuntos = ['test', 'train_val']
    
    print("\n" + "="*60)
    print("SISTEMA DE SINCRONIZACIÓN INTERACTIVA")
    print("="*60)

    for sub in subconjuntos:
        sub_path = os.path.join(output_root, sub)
        if not os.path.exists(sub_path):
            continue

        vistas = sorted([d for d in os.listdir(sub_path) if os.path.isdir(os.path.join(sub_path, d))])
        if not vistas:
            continue

        # --- FASE 1: DETECCIÓN (Lógica idéntica a tu auditoría) ---
        vistas_data = {}
        vistas_files = {}
        
        print(f"\n[1] Analizando {sub.upper()}...")
        
        for vista in vistas:
            vista_dir = os.path.join(sub_path, vista)
            timestamps_en_vista = set()
            file_map = {}
            
            for f in os.listdir(vista_dir):
                if f.endswith('.csv'):
                    partes = f.split('_')
                    if len(partes) >= 3:
                        ts_id = "_".join(partes[2:])
                        timestamps_en_vista.add(ts_id)
                        file_map[ts_id] = f
            
            vistas_data[vista] = timestamps_en_vista
            vistas_files[vista] = file_map
            print(f"    -> Vista {vista}: {len(timestamps_en_vista)} archivos detectados.")

        # Calculamos la intersección real
        common_timestamps = set.intersection(*vistas_data.values())
        
        # --- FASE 2: MOSTRAR DISCREPANCIAS ---
        print(f"\n[2] RESULTADOS DE COMPARACIÓN:")
        print(f"    - Frames comunes que SE QUEDAN: {len(common_timestamps)}")
        
        total_a_borrar = 0
        huerfanos_por_vista = {}
        
        for vista in vistas:
            huerfanos = vistas_data[vista] - common_timestamps
            huerfanos_por_vista[vista] = huerfanos
            if huerfanos:
                print(f"    - Vista {vista}: se borrarán {len(huerfanos)} archivos huérfanos.")
                total_a_borrar += len(huerfanos)

        # --- FASE 3: CONFIRMACIÓN Y ACCIÓN ---
        if total_a_borrar > 0:
            print(f"\n[!] ATENCIÓN: Se van a eliminar un total de {total_a_borrar} archivos en {sub.upper()}.")
            confirmacion = input(f"    ¿Deseas borrar estos archivos para sincronizar {sub.upper()}? (s/n): ").lower()
            
            if confirmacion == 's':
                print(f"    Iniciando borrado...")
                for vista, ids_borrar in huerfanos_por_vista.items():
                    vista_dir = os.path.join(sub_path, vista)
                    for ts_id in ids_borrar:
                        nombre_real = vistas_files[vista][ts_id]
                        os.remove(os.path.join(vista_dir, nombre_real))
                print(f"    [OK] {sub.upper()} sincronizado correctamente.")
            else:
                print(f"    [Sancela] No se han realizado cambios en {sub.upper()}.")
        else:
            print(f"    [OK] El subconjunto {sub.upper()} ya está perfectamente sincronizado.")

    print("\n" + "="*60)
    print("PROCESO FINALIZADO")
    print("="*60)

if __name__ == "__main__":
    # Ejecuta esta función para tener control total
    sync_and_clean_interactive("Output_CSVs_backup/")


SISTEMA DE SINCRONIZACIÓN INTERACTIVA

[1] Analizando TEST...
    -> Vista 00_15: 3153 archivos detectados.
    -> Vista 00_16: 3151 archivos detectados.
    -> Vista 00_17: 3155 archivos detectados.

[2] RESULTADOS DE COMPARACIÓN:
    - Frames comunes que SE QUEDAN: 3149
    - Vista 00_15: se borrarán 4 archivos huérfanos.
    - Vista 00_16: se borrarán 2 archivos huérfanos.
    - Vista 00_17: se borrarán 6 archivos huérfanos.

[!] ATENCIÓN: Se van a eliminar un total de 12 archivos en TEST.


    ¿Deseas borrar estos archivos para sincronizar TEST? (s/n):  s


    Iniciando borrado...
    [OK] TEST sincronizado correctamente.

[1] Analizando TRAIN_VAL...
    -> Vista 00_15: 15162 archivos detectados.
    -> Vista 00_16: 15029 archivos detectados.
    -> Vista 00_17: 15256 archivos detectados.

[2] RESULTADOS DE COMPARACIÓN:
    - Frames comunes que SE QUEDAN: 14868
    - Vista 00_15: se borrarán 294 archivos huérfanos.
    - Vista 00_16: se borrarán 161 archivos huérfanos.
    - Vista 00_17: se borrarán 388 archivos huérfanos.

[!] ATENCIÓN: Se van a eliminar un total de 843 archivos en TRAIN_VAL.


    ¿Deseas borrar estos archivos para sincronizar TRAIN_VAL? (s/n):  s


    Iniciando borrado...
    [OK] TRAIN_VAL sincronizado correctamente.

PROCESO FINALIZADO
